# Annual Report Question-Answering Assistant (RAG)

**A retrieval-augmented generation (RAG) project over the Infosys Integrated Annual Report 2023-24 (353 pages).**

I built an assistant that answers plain-English questions about a long company report and shows the **file and page** each answer came from. When the report does not contain the answer, it says so instead of guessing.

## Project summary

| | |
|---|---|
| **Problem** | Annual reports run to hundreds of pages, so finding one figure by hand is slow, and a plain chatbot may invent numbers. |
| **Approach** | Split the report into chunks, retrieve the relevant ones with hybrid search plus a re-ranker, and let a language model answer using only those chunks. |
| **Data** | Infosys Integrated Annual Report 2023-24 (NSE archive PDF): 353 pages, 2,949 chunks. |
| **Stack** | Python, sentence-transformers (bge-small), rank_bm25, a cross-encoder re-ranker, the Gemini API, Google Colab. |
| **Evaluation** | A 12-question development set, 10 fresh questions and 2 questions the report cannot answer. |

## Results at a glance

- **Search:** hybrid search plus a re-ranker put the right chunk in the top 3 for **11 of 12** development questions (meaning-only search managed 9 of 12).
- **Answers:** **12 of 12** development questions answered correctly with Gemini, against about 9 of 12 with a free local model.
- **Fresh questions I did not tune against:** right chunk in the top 5 for **7 of 10**, correct answers for **7 of 10**, and **both out-of-scope questions were refused correctly**. Every miss was a refusal or a different reading of the question, not a confident wrong answer.

The fresh-question scores are the honest ones: the development set is partly tuned (see Step 9).

## How the pipeline works

```
Upload  ->  Chunk  ->  Embed  ->  Hybrid search  ->  Re-rank  ->  Answer model  ->  Answer + sources
                                  (meaning + keywords)  (reads question and chunk together)
```

## Contents

| Step | What it does |
|---|---|
| 1 | Install libraries |
| 2 | Imports and settings |
| 3 | Upload the document |
| 4 | Chunking |
| 5 | Embeddings |
| 6 | Retrieval: dense, hybrid and re-ranked search |
| 7 | The answer model (Gemini API, with a local fallback) |
| 8 | The RAG function `ask()` |
| 9 | Evaluation: development set, fresh set and what the scores mean |
| 10 | Sample questions |
| 11 | Chat mode |
| 12 | Limitations, what I learned and next steps |

## Run it yourself

1. **Platform:** Google Colab (free). No GPU needed.
2. **Gemini API key:** create one in Google AI Studio. In Colab open the **key icon (Secrets)**, add a secret named exactly `LLM_API_KEY`, paste the key and switch on **Notebook access**. Never paste a key into a cell.
3. **Document:** the report PDF from the NSE archive (link in the repository README). The test sets in Step 9 are written for this report; for another document, replace the questions.
4. **No key?** Set `ANSWER_BACKEND = "local"` in Step 2 to use a free local model (slower and less accurate).
5. Run the cells **one by one, in order, with Shift+Enter**. Do not use *Run all*: Step 3 needs a file upload and Step 11 waits for typing.

**Good to know:** free Gemini models allow only a small number of requests per day (I hit 20 per day on one model), so the notebook saves answers it already has and stops cleanly when the quota runs out. Colab also forgets everything when a session ends, so re-run Steps 1 to 8 after a break.

## Step 1: Install libraries

| Library | Used for |
|---|---|
| `sentence-transformers` | embeddings and the re-ranker |
| `rank_bm25` | keyword search (BM25) |
| `pypdf` | reading PDF text |
| `google-genai` | calling the Gemini API |

*Pip may print red "dependency conflict" warnings about `google-auth` or `google-adk`. They do not affect this notebook.*

In [1]:
!pip install -q sentence-transformers transformers torch numpy pypdf rank_bm25 google-genai

## Step 2: Imports and settings

The settings I used are collected in one cell so they are easy to change.

| Setting | Value I used | Meaning |
|---|---|---|
| `ANSWER_BACKEND` | `"gemini"` | `"gemini"` (API) or `"local"` (free `flan-t5-large`, weaker answers) |
| `GEMINI_MODEL` | `gemini-3.1-flash-lite` | the model that writes answers; every model has its own free daily limit |
| `EMBEDDING_MODEL` | `BAAI/bge-small-en-v1.5` | turns text into vectors |
| `TOP_K` | `5` | chunks given to the answer model (5 worked better than 3 in my tests) |

In [2]:
import io
import os
import re
import time
import numpy as np
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

# ---------------- Settings (the values I used) ----------------
ANSWER_BACKEND = "gemini"                  # "gemini" (needs the LLM_API_KEY secret) or "local" (free, no key)
GEMINI_MODEL = "gemini-3.1-flash-lite"     # a light model; each model has its own free daily limit
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5" # small embedding model that runs on CPU (about 130 MB)
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
TOP_K = 5                                  # chunks sent to the answer model (5 beat 3 in my tests)
# ----------------------------------------------------------

print("Libraries loaded. Answer backend:", ANSWER_BACKEND)

Libraries loaded. Answer backend: gemini


## Step 3: Upload the document

I upload the report PDF here. Each PDF page is kept separately so every answer can cite its page number. *(The page number is the position in the PDF file, which can differ from the number printed on the page.)*

- Text-based PDFs, TXT and MD files work. Scanned image-only PDFs have no text to read, and the cell warns about them.
- Not on Colab? Put the files in a folder named `docs` next to the notebook and the cell reads from there.

In [3]:
def read_document(name, data):
    '''Return a list of (source, page, text) for one file.'''
    parts = []
    if name.lower().endswith(".pdf"):
        reader = PdfReader(io.BytesIO(data))
        for i, page in enumerate(reader.pages):
            text = (page.extract_text() or "").strip()
            if text:                      # skip empty pages (images only)
                parts.append((name, i + 1, text))
    else:
        text = data.decode("utf-8", errors="ignore").strip()
        if text:
            parts.append((name, None, text))
    return parts


uploaded = {}
try:
    from google.colab import files        # only exists inside Colab
    uploaded = files.upload()
except ImportError:
    if os.path.isdir("docs"):
        for fn in os.listdir("docs"):
            with open(os.path.join("docs", fn), "rb") as f:
                uploaded[fn] = f.read()
    else:
        print("No 'docs' folder found. Create one and add your files.")

pages = []
for name, data in uploaded.items():
    got = read_document(name, data)
    if not got:
        print(f"WARNING: no readable text in {name} (scanned PDF?)")
    pages.extend(got)

print(f"\nLoaded {len(uploaded)} file(s), {len(pages)} page(s)/section(s) of text")

Saving INFY_03062024153620_infosysar24.pdf to INFY_03062024153620_infosysar24.pdf

Loaded 1 file(s), 353 page(s)/section(s) of text


## Step 4: Chunking

A 353-page report is far too long to give to a language model, so I cut it into small **overlapping chunks** of about 80 words (20 words shared with the next chunk). Search can then find the few chunks that matter.

**Trade-off I noticed:** small chunks keep search precise, but a sentence or table can get cut in half. The overlap reduces that. It also explains why numbers in a table are sometimes separated from their labels, which is the hardest case for the answer model.

In [4]:
def chunk_words(text, size=80, overlap=20):
    '''Split text into chunks of `size` words, each sharing `overlap` words with the previous one.'''
    words = text.split()
    step = size - overlap
    out = []
    for start in range(0, len(words), step):
        piece = words[start:start + size]
        if len(piece) < 15 and start > 0:   # drop a tiny leftover tail
            break
        out.append(" ".join(piece))
    return out


chunks = []
for source, page, text in pages:
    for piece in chunk_words(text):
        chunks.append({"source": source, "page": page, "text": piece})

print(f"Created {len(chunks)} chunks")
assert chunks, "No chunks. Go back to Step 3 and upload documents with readable text."
print("\nExample chunk:\n", chunks[0])

Created 2949 chunks

Example chunk:
 {'source': 'INFY_03062024153620_infosysar24.pdf', 'page': 1, 'text': 'TO ALL STOCK EXCHANGES BSE LIMITED NATIONAL STOCK EXCHANGE OF INDIA LIMITED NEW YORK STOCK EXCHANGE June 3, 2024 Dear Sir/ Madam, Sub: Notice of the 43rd Annual General Meeting (AGM) and Integrated Annual Report 2023-24 In compliance with the Companies Act 2013, rules framed thereunder and SEBI (Listing Obligations and Disclosure Requirements) Regulations, 2015, as amended from time to time, please find enclosed Notice convening the 43rd AGM of shareholders and the Integrated Annual Report 2023-24 which will be'}


## Step 5: Embeddings (turn text into vectors)

An embedding model turns each chunk into a list of numbers so that chunks with **similar meaning** end up close together. I use `BAAI/bge-small-en-v1.5`, a small model that runs on CPU (about 130 MB). Embedding the whole report takes a couple of minutes.

I also tried `all-MiniLM-L6-v2`. The two scored about the same on this report, so I kept bge-small.

In [5]:
embedder = SentenceTransformer(EMBEDDING_MODEL)

chunk_vectors = embedder.encode(
    [c["text"] for c in chunks],
    normalize_embeddings=True,     # unit-length vectors: dot product = cosine similarity
    show_progress_bar=True,
)
print("Embedding matrix shape:", chunk_vectors.shape)   # (number of chunks, vector size)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/93 [00:00<?, ?it/s]

Embedding matrix shape: (2949, 384)


## Step 6: Retrieval

Retrieval finds the chunks most likely to contain the answer. I built **three versions** so that Step 9 can measure what each one adds.

| Version | Idea | Strength | Weakness |
|---|---|---|---|
| Dense | search by meaning | handles different wording | misses exact words and numbers |
| Hybrid | dense + keyword (BM25), merged | finds exact terms like "interim dividend" | ranking still rough |
| Hybrid + re-ranker | a second model re-reads question and chunk together | most accurate ordering | slower |

### 6a. Dense search (search by meaning)

In [6]:
def retrieve_dense(query, k=TOP_K):
    q_vec = embedder.encode([query], normalize_embeddings=True)[0]
    scores = chunk_vectors @ q_vec                 # cosine similarity with every chunk
    top_idx = np.argsort(scores)[::-1][:k]         # best k
    return [(chunks[i], float(scores[i])) for i in top_idx]

print("Dense search ready")

Dense search ready


### 6b. Hybrid search (meaning + keywords)

Meaning-based search is weak on exact words and numbers ("attrition", "interim dividend"), and keyword search (**BM25**) is strong on them. I run both and merge the two ranked lists with **reciprocal rank fusion**, so a chunk that ranks high in either list rises to the top.

In [7]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])   # keyword index


def retrieve_hybrid(query, k=TOP_K, pool=30, rrf_k=60):
    # ranking 1: by meaning
    q_vec = embedder.encode([query], normalize_embeddings=True)[0]
    dense_scores = chunk_vectors @ q_vec
    dense_rank = np.argsort(dense_scores)[::-1][:pool]

    # ranking 2: by keywords
    bm_scores = bm25.get_scores(tokenize(query))
    bm_rank = np.argsort(bm_scores)[::-1][:pool]

    # merge: each list gives a chunk 1/(rrf_k + rank) points
    fused = {}
    for r, i in enumerate(dense_rank):
        fused[i] = fused.get(i, 0) + 1 / (rrf_k + r + 1)
    for r, i in enumerate(bm_rank):
        fused[i] = fused.get(i, 0) + 1 / (rrf_k + r + 1)

    top = sorted(fused, key=fused.get, reverse=True)[:k]
    return [(chunks[i], float(dense_scores[i])) for i in top]

print("Hybrid search ready")

Hybrid search ready


### 6c. Re-ranker (read the question and chunk together)

The hybrid search pulls **20 candidate chunks**. A **cross-encoder** then reads the question together with each candidate and re-sorts them, so only the best few go to the answer model. It is slower, but much better at putting the right chunk first. The cell ends by making this the search used by the rest of the notebook.

In [8]:
reranker = CrossEncoder(RERANK_MODEL)


def retrieve_reranked(query, k=TOP_K, pool=20):
    candidates = retrieve_hybrid(query, k=pool)                  # step 1: 20 candidates
    pairs = [(query, chunk["text"]) for chunk, _ in candidates]
    scores = reranker.predict(pairs)                             # step 2: score each (question, chunk) pair
    order = np.argsort(scores)[::-1][:k]
    return [(candidates[i][0], float(candidates[i][1])) for i in order]


# The pipeline below uses this version. Switch to retrieve_hybrid or retrieve_dense to compare.
retrieve = retrieve_reranked
print("Re-ranker ready. Active search: hybrid + re-ranker")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Re-ranker ready. Active search: hybrid + re-ranker


## Step 7: The answer model

After retrieval, a language model reads the best chunks and writes the answer. I tried two options, chosen by `ANSWER_BACKEND` in Step 2:

| Backend | Pros | Cons |
|---|---|---|
| `gemini` (the one I use) | much more accurate, copes with messy tables | needs an API key; the free tier has a small daily request limit |
| `local` (`flan-t5-large`) | free, no key | 3 GB download; often picked the wrong number or refused |

### 7a. Connect to Gemini
The key is read from **Colab Secrets** (secret name `LLM_API_KEY`, Notebook access on) and is never printed.

In [9]:
client = None

if ANSWER_BACKEND == "gemini":
    from google import genai
    from google.genai import types

    # read the key from Colab Secrets (or from an environment variable outside Colab)
    try:
        from google.colab import userdata
        GEMINI_API_KEY = userdata.get("LLM_API_KEY")
    except ImportError:
        GEMINI_API_KEY = os.environ.get("LLM_API_KEY")
    except Exception as e:                      # secret missing, or Notebook access switched off
        GEMINI_API_KEY = None
        print("Could not read the secret LLM_API_KEY:", type(e).__name__)

    if not GEMINI_API_KEY:
        raise RuntimeError(
            "No API key found. Open the key icon (Secrets) in the left sidebar, add a secret named "
            "LLM_API_KEY, paste your key and switch on Notebook access. Or set ANSWER_BACKEND = 'local' in Step 2."
        )

    # timeout: give up on a single request after 30 seconds instead of hanging
    client = genai.Client(api_key=GEMINI_API_KEY, http_options=types.HttpOptions(timeout=30000))
    print("Connected to Gemini. Model:", GEMINI_MODEL)
else:
    print("Using the local model (it loads the first time it is used).")

Connected to Gemini. Model: gemini-3.1-flash-lite


### 7b. The `generate()` function

One function that every later step calls, whichever backend is selected.

- **Quota protection:** if Gemini says the free daily quota is used up (error 429), the code raises `QuotaStop`, so loops stop immediately instead of wasting requests.
- **Busy servers:** 503 and 504 errors ("high demand", "deadline exceeded") come from Google's side and are usually temporary, so the code retries once after a short wait.

In [10]:
class QuotaStop(Exception):
    '''Raised when the free daily quota is used up, so loops can stop instead of wasting requests.'''


def generate_gemini(prompt):
    for attempt in (1, 2):
        try:
            r = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
            return (r.text or "").strip()
        except Exception as e:
            msg = str(e)
            if "429" in msg or "RESOURCE_EXHAUSTED" in msg:      # quota / rate limit
                raise QuotaStop(msg[:300])
            if attempt == 2:                                      # still failing: report, don't crash
                return f"ERROR: {msg[:100]}"
            time.sleep(8)                                         # 503/504: wait, then retry once


def generate_local(prompt, max_new_tokens=120):
    '''Free fallback: google/flan-t5-large on CPU (downloads about 3 GB the first time).'''
    global gen_tokenizer, gen_model
    if "gen_model" not in globals():
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        print("Loading local model (first time only)...")
        gen_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
        gen_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")
        gen_model.eval()
    import torch
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        out = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return gen_tokenizer.decode(out[0], skip_special_tokens=True).strip()


def generate(prompt):
    return generate_gemini(prompt) if ANSWER_BACKEND == "gemini" else generate_local(prompt)

print("generate() ready, backend:", ANSWER_BACKEND)

generate() ready, backend: gemini


### 7c. Smoke test (uses 1 request)

Checks that the answer model responds. If it fails:

| Error | Meaning | What I do |
|---|---|---|
| `429` / quota | free daily limit reached for this model | try another model (7d) or wait for the reset |
| `503` | Google's model is busy | run again in a minute |
| `504` | request timed out | run again |
| `API key not valid` | wrong key | recreate the key and update the secret |

In [ ]:
try:
    print("Model reply:", generate("Reply with the single word: ready"))
except QuotaStop as e:
    print("Daily quota used up for", GEMINI_MODEL, "- see 7d below.\n", e)

Model reply: ready


### 7d. Troubleshooting (optional, run only if 7c failed)

I needed this once: `gemini-3.8-flash` allowed only **20 requests per day** on the free tier and my first evaluation run used them up. Limits are counted **per model**, so this cell lists the models my key can use, and I switched to `gemini-3.1-flash-lite`. To switch, copy a model name (without the `models/` prefix) into `GEMINI_MODEL` in Step 2 and re-run Steps 2 and 7a.

In [ ]:
# OPTIONAL: run only if Step 7c failed. Listing models does not use your answer quota.
for m in client.models.list():
    if "gemini" in m.name and "embed" not in m.name:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/gemini-4-carbon
models/gemini-4-hydrogen
models/gemini-4-neon
models/gemini-3.1-flash-tts-preview
models/g

## Step 8: The RAG function

`ask()` is the whole system in one call:

1. **retrieve** the best chunks for the question
2. **build a prompt** that tells the model to answer *only* from those chunks, and to say "I don't have that information" otherwise
3. **generate** the answer
4. **print the sources** (file and page)

There are two safety nets against made-up answers: the instruction in the prompt, and `MIN_SCORE`, which refuses to answer when even the best chunk is only weakly related to the question.

In [11]:
MIN_SCORE = 0.15   # loose safety net; the "say you don't know" instruction in the prompt does most of the work


def build_prompt(question, results):
    '''Put the retrieved chunks and the question into one instruction for the model.'''
    context = "\n".join(f"- {chunk['text']}" for chunk, _ in results)
    return (
        "Answer the question using only the context. "
        "If the answer is not in the context, say 'I don't have that information.'\n\n"
        f"Question: {question}\n\n"
        f"Context:\n{context}\n\n"
        "Answer:"
    )


def get_answer(question, k=TOP_K):
    '''Retrieve + generate, return only the answer text (used by the evaluation).'''
    results = retrieve(question, k=k)
    return generate(build_prompt(question, results))


def ask(question, k=TOP_K):
    '''Retrieve + generate, then print the question, answer and sources.'''
    results = retrieve(question, k=k)
    best_score = results[0][1]

    print(f"Q: {question}")
    if best_score < MIN_SCORE:
        print("A: I couldn't find anything relevant in the uploaded documents.")
        print("-" * 60)
        return

    try:
        print("A:", generate(build_prompt(question, results)))
    except QuotaStop as e:
        print("A: [STOPPED] The free daily quota for this model is used up.", str(e)[:150])
        print("-" * 60)
        return

    seen = []                                       # unique "file (page N)" labels, in rank order
    for chunk, score in results:
        label = chunk["source"] + (f" (page {chunk['page']})" if chunk["page"] else "")
        if label not in seen:
            seen.append(label)
    print("Sources:", "; ".join(seen))
    print("-" * 60)

print("RAG function ready")

RAG function ready


## Step 9: Evaluation (development set, fresh set and what the scores mean)

A RAG project is only convincing if it is **measured**. I used two test sets:

1. **Development set (12 questions)**, used while building the pipeline. I drafted it from the report with AI assistance and checked it against the report text.
2. **Fresh set (10 questions + 2 out-of-scope)**, drafted after the pipeline was finished and **not used to tune anything** (Step 9c).

Each question has the **expected answer**, **`keys`** (text that must appear in a retrieved chunk for the *search* to count as correct, checked automatically) and **`answer_keys`** (text that should appear in the *answer*, a rough automatic check).

**Search metrics:** **hit@1** = the right chunk was the top result, **hit@3** = it was in the top 3, **hit@5** = it was in the top 5 (the answer model reads the top 5).

In [12]:
test_set = [
    # q = question | expected = correct answer (for me to read)
    # keys = text the retrieved chunk must contain | answer_keys = text the answer should contain
    {"q": "What were the total revenues in fiscal 2024?",
     "expected": "Rs 1,53,670 crore (US$18,562 million)",
     "keys": ["1,53,670"], "answer_keys": ["18,562", "1,53,670"]},
    {"q": "How many employees does Infosys have?",
     "expected": "3,17,240 (over 3,17,000)",
     "keys": ["3,17,240", "3,17,000"], "answer_keys": ["3,17,240", "3,17,000"]},
    {"q": "What was the employee attrition rate?",
     "expected": "12.6%",
     "keys": ["12.6%"], "answer_keys": ["12.6"]},
    {"q": "What final dividend per share was recommended?",
     "expected": "Rs 20 per share",
     "keys": ["final dividend of 20"], "answer_keys": ["20per", "20/-"]},
    {"q": "What interim dividend per share was paid?",
     "expected": "Rs 18 per share",
     "keys": ["interim dividend of 18"], "answer_keys": ["18per"]},
    {"q": "What special dividend per share was recommended?",
     "expected": "Rs 8 per share",
     "keys": ["special dividend of 8"], "answer_keys": ["8per", "8.00"]},
    {"q": "When is the 43rd Annual General Meeting?",
     "expected": "June 26, 2024, 4:00 p.m. IST",
     "keys": ["june 26, 2024"], "answer_keys": ["june26,2024"]},
    {"q": "What is the record date for the final dividend?",
     "expected": "May 31, 2024",
     "keys": ["may 31, 2024"], "answer_keys": ["may31,2024"]},
    {"q": "By which year is the Net Zero planet goal under the Climate Pledge?",
     "expected": "2040",
     "keys": ["net zero planet"], "answer_keys": ["2040"]},
    {"q": "How much did Infosys win in large deals?",
     "expected": "US$17.7 billion",
     "keys": ["17.7 billion"], "answer_keys": ["17.7"]},
    {"q": "How many employees were trained on AI skills?",
     "expected": "over 2,50,000",
     "keys": ["2,50,000 employees on ai"], "answer_keys": ["2,50,000"]},
    {"q": "Who is the Company Secretary?",
     "expected": "A.G.S. Manikantha",
     "keys": ["manikantha"], "answer_keys": ["manikantha"]},
]


def norm(s):
    '''Ignore spaces, capitals and currency symbols so small PDF-extraction differences don't count as misses.'''
    return "".join(s.split()).lower().replace("₹", "").replace("`", "")


def contains_key(text, keys):
    t = norm(text)
    return any(norm(k) in t for k in keys)


def rank_of_right_chunk(retriever, item, k=5):
    '''Position (1..k) of the first retrieved chunk that contains a key, or None.'''
    for r, (chunk, _) in enumerate(retriever(item["q"], k=k), start=1):
        if contains_key(chunk["text"], item["keys"]):
            return r
    return None

print(f"Test set ready: {len(test_set)} questions")

Test set ready: 12 questions


### 9a. Compare the three search versions (free: uses no API requests)

In [ ]:
n = len(test_set)
print(f"{'Search version':<24}{'hit@1':>8}{'hit@3':>8}{'hit@5':>8}")
print("-" * 48)

for name, fn in [
    ("Dense (meaning only)", retrieve_dense),
    ("Hybrid (+ keywords)", retrieve_hybrid),
    ("Hybrid + re-ranker", retrieve_reranked),
]:
    ranks = [rank_of_right_chunk(fn, item) for item in test_set]
    h = {k: sum(1 for r in ranks if r is not None and r <= k) for k in (1, 3, 5)}
    print(f"{name:<24}{str(h[1]) + '/' + str(n):>8}{str(h[3]) + '/' + str(n):>8}{str(h[5]) + '/' + str(n):>8}")

Search version             hit@1   hit@3   hit@5
------------------------------------------------
Dense (meaning only)        6/12    9/12    9/12
Hybrid (+ keywords)         6/12    9/12   11/12
Hybrid + re-ranker          8/12   11/12   12/12


#### What the search results show

| Search version | hit@1 | hit@3 | hit@5 |
|---|---|---|---|
| Dense (meaning only) | 6/12 | 9/12 | 9/12 |
| Hybrid (+ keywords) | 6/12 | 9/12 | 11/12 |
| Hybrid + re-ranker | **8/12** | **11/12** | **12/12** |

- **Keywords help recall.** Adding BM25 lifted hit@5 from 9 to 11. It rescued questions that depend on exact terms such as "interim dividend" and "attrition", which pure meaning search missed.
- **The re-ranker fixes the ordering.** It pushed the right chunk higher (hit@3 from 9 to 11), and the right chunk was in the top 5 for every development question. That matters because the answer model can only use the chunks it is given.
- **Caution:** with 12 questions, one question is about 8 percentage points. I treat these numbers as indicative, not proof.

### 9b. Review the answers (uses up to 12 requests)

This runs the full pipeline on every development question and shows, for each: where the right chunk ranked, the expected answer, what the model said, and a rough automatic check (`OK` = the expected value appears in the answer).

- **Answers are saved** in `answer_cache`, so re-running the cell never pays for the same question twice.
- **If the daily quota runs out**, it stops, keeps what it already has, and can be re-run later to continue.
- The automatic check is only a hint: an answer can contain the right value and still be partly wrong, so I also read every answer myself.

In [ ]:
if "answer_cache" not in globals():
    answer_cache = {}                     # question -> answer; survives re-runs of this cell


def review_answers():
    checked = matched = 0
    for item in test_set:
        q = item["q"]

        if q in answer_cache:             # already answered: no new request
            ans = answer_cache[q]
        else:
            try:
                ans = get_answer(q)
            except QuotaStop as e:
                print("STOPPED: free quota reached. Re-run this cell later to continue.\n", str(e)[:200])
                break
            if not ans.startswith("ERROR"):
                answer_cache[q] = ans

        rank = rank_of_right_chunk(retrieve, item)
        ok = contains_key(ans, item["answer_keys"])
        checked += 1
        matched += ok

        print(f"Q: {q}")
        print(f"   right chunk found at rank: {rank if rank else 'NOT in top 5'}")
        print(f"   expected: {item['expected']}")
        print(f"   model said: {ans[:140]}")
        print(f"   auto-check: {'OK' if ok else 'CHECK THIS ONE'}")

    print(f"\nAuto-check: {matched}/{checked} answers contain the expected value  (backend: {ANSWER_BACKEND}, model: {GEMINI_MODEL if ANSWER_BACKEND == 'gemini' else 'flan-t5-large'})")

review_answers()

Q: What were the total revenues in fiscal 2024?
   right chunk found at rank: 3
   expected: Rs 1,53,670 crore (US$18,562 million)
   model said: The total revenues in fiscal 2024 were US$ 18,562 million or ₹ 1,53,670 crore (consolidated).
   auto-check: OK
Q: How many employees does Infosys have?
   right chunk found at rank: 1
   expected: 3,17,240 (over 3,17,000)
   model said: As of March 31, 2024, Infosys had 2,48,297 employees on a standalone basis and 3,17,240 employees on a consolidated basis. Another section o
   auto-check: OK
Q: What was the employee attrition rate?
   right chunk found at rank: 3
   expected: 12.6%
   model said: The employee attrition rate is 12.6%.
   auto-check: OK
Q: What final dividend per share was recommended?
   right chunk found at rank: 1
   expected: Rs 20 per share
   model said: The final dividend recommended was ₹20 per equity share.
   auto-check: OK
Q: What interim dividend per share was paid?
   right chunk found at rank: 1
   expected: Rs 

#### What the answer results show

| Answer model | Chunks given | Answers right |
|---|---|---|
| `flan-t5-large` (free, local) | top 3 | about 9 / 12 (refused revenue and employees, gave a wrong number for large deals) |
| Gemini (flash-lite) | top 3 | 11 / 12 (the 12th was an honest "I don't have that information") |
| Gemini (flash-lite) | top 5 | 12 / 12 |

- **After the re-ranker, the answer model was the bottleneck, not search.** The right chunk was found in every case, but the small local model could not tell which number in a jumbled table belonged to which label. A stronger model fixed most of it.
- **Giving 5 chunks instead of 3 fixed the last miss** (the large-deals chunk ranked 4th). I made that change *after* seeing the failure, so the 12/12 is partly tuned to this set. That is why I also ran the fresh set below.
- **Refusing is a feature.** A system that says "I don't have that information" when the evidence is missing is safer than one that guesses.

### 9c. Fresh test set (10 unseen questions + 2 out-of-scope)

These questions were drafted after the pipeline was finished and were **not used to tune it**, so they give a more honest score than 9b. They cover parts of the report the development set did not (leadership, auditors, margins, regions, clients, hiring). This cell uses about 12 API requests.

| Measure | Result |
|---|---|
| Right chunk in top 3 | 5/10 |
| Right chunk in top 5 | 7/10 |
| Answers matching | 7/10 |
| Out-of-scope questions refused correctly | 2/2 |

**What I found**
- **Two misses were retrieval failures** (free cash flow and active clients). Those figures sit in highlight boxes and stat tiles, which the search handles poorly. The model answered "I don't have that information" rather than inventing a number.
- **One miss was a wording ambiguity.** Asked how much revenue grew "in rupee terms", the model gave the rupee increase (about ₹6,903 crore, which I checked is 1,53,670 minus 1,46,767) instead of the 4.7% figure I expected. The auto-check marked it wrong and I kept it that way.
- **There were no confident wrong answers.** The women-in-workforce answer of 39.3% is the report's exact figure (the report text also says "about 39%").
- **I did not change the pipeline to fix these three.** Doing so would turn the fresh set into another tuned set. Any later improvement should be checked on a new set.

In [13]:
# Step 9c: fresh test questions, never used while building the pipeline
fresh_test_set = [
    {"q": "Who is the Chairman of Infosys?", "expected": "Nandan M. Nilekani",
     "keys": ["nilekani, chairman", "nilekani chairman"], "answer_keys": ["nilekani"]},
    {"q": "Who is the statutory auditor of Infosys?", "expected": "Deloitte Haskins & Sells LLP",
     "keys": ["deloitte haskins"], "answer_keys": ["deloitte"]},
    {"q": "What was the operating margin in fiscal 2024?", "expected": "20.7%",
     "keys": ["20.7%"], "answer_keys": ["20.7"]},
    {"q": "By how much did revenues grow year on year in rupee terms?", "expected": "4.7%",
     "keys": ["4.7% growth"], "answer_keys": ["4.7"]},
    {"q": "What was the free cash flow?", "expected": "Rs 23,865 crore (about US$2.9 billion)",
     "keys": ["23,865", "2.9 billion"], "answer_keys": ["23,865", "2.9"]},
    {"q": "What share of revenue came from North America?", "expected": "60.1%",
     "keys": ["north america 60.1%"], "answer_keys": ["60.1"]},
    {"q": "How many active clients did Infosys have at year end?", "expected": "1,882",
     "keys": ["1,882"], "answer_keys": ["1,882"]},
    {"q": "How many new clients were added during the year?", "expected": "385",
     "keys": ["385 new clients"], "answer_keys": ["385"]},
    {"q": "How many college graduates did Infosys recruit?", "expected": "nearly 11,900",
     "keys": ["11,900"], "answer_keys": ["11,900"]},
    {"q": "What share of the workforce are women?", "expected": "about 39%",
     "keys": ["about 39%"], "answer_keys": ["39"]},
]
out_of_scope = [
    "What is the Infosys share price target for 2025?",
    "Who won the cricket world cup?",
]
refusal_keys = ["havethatinformation", "notcontain", "notmention", "notprovided", "cannotfind", "couldnotfind"]

if "fresh_answers" not in globals():
    fresh_answers = {}

def run_fresh_tests():
    n = len(fresh_test_set)
    hit3 = hit5 = ok = 0
    for item in fresh_test_set:
        q = item["q"]
        if q in fresh_answers:
            ans = fresh_answers[q]
        else:
            try:
                ans = get_answer(q)
            except QuotaStop as e:
                print("STOPPED: free quota reached. Re-run later.\n", str(e)[:150]); return
            if not ans.startswith("ERROR"):
                fresh_answers[q] = ans
        rank = rank_of_right_chunk(retrieve, item)
        good = contains_key(ans, item["answer_keys"])
        hit3 += bool(rank and rank <= 3); hit5 += bool(rank)
        ok += good
        print(f"Q: {q}\n   right chunk rank: {rank if rank else 'NOT in top 5'}")
        print(f"   expected: {item['expected']}\n   model said: {ans[:130]}\n   auto-check: {'OK' if good else 'CHECK THIS ONE'}")
    print(f"\nFRESH SET -> hit@3: {hit3}/{n}  hit@5: {hit5}/{n}  answers matching: {ok}/{n}")

    print("\nQuestions the report cannot answer (should refuse):")
    for q in out_of_scope:
        try:
            ans = fresh_answers.get(q) or get_answer(q)
        except QuotaStop:
            print("STOPPED: quota reached."); return
        fresh_answers[q] = ans
        print(f"Q: {q}\n   model said: {ans[:130]}\n   refused: {'YES' if contains_key(ans, refusal_keys) else 'NO, READ IT'}")

run_fresh_tests()

Q: Who is the Chairman of Infosys?
   right chunk rank: 1
   expected: Nandan M. Nilekani
   model said: Nandan M. Nilekani is the Chairman of Infosys.
   auto-check: OK
Q: Who is the statutory auditor of Infosys?
   right chunk rank: 1
   expected: Deloitte Haskins & Sells LLP
   model said: The statutory auditor of Infosys is Deloitte Haskins & Sells LLP.
   auto-check: OK
Q: What was the operating margin in fiscal 2024?
   right chunk rank: 1
   expected: 20.7%
   model said: The operating margin in fiscal 2024 was 20.7%.
   auto-check: OK
Q: By how much did revenues grow year on year in rupee terms?
   right chunk rank: NOT in top 5
   expected: 4.7%
   model said: Based on the provided context, the consolidated revenues grew by ₹6,903 crore (calculated as 1,53,670 - 1,46,767) and the standalo
   auto-check: CHECK THIS ONE
Q: What was the free cash flow?
   right chunk rank: NOT in top 5
   expected: Rs 23,865 crore (about US$2.9 billion)
   model said: I don't have that informatio

## Step 10: Sample questions

A few free-form questions beyond the test sets, answered by the full pipeline. **Each question uses one API request.** The last question is deliberately **not** in the report: a good system should say it does not know instead of inventing an answer. I check the page numbers under *Sources* each time.

*Worth noting from the net-profit question:* the answer lists three figures (₹26,248 crore consolidated profit for the year, ₹26,233 crore attributable to owners, ₹27,234 crore standalone). I checked all three in the report, and they are correct. The headline "net profit" in the five-year summary is ₹26,233 crore.

In [ ]:
sample_questions = [
    "What was the net profit for the year?",
    "Who is the Chief Executive Officer?",
    "What are the main risks faced by the company?",
    "What is the company's plan for generative AI?",
    "What is the weather in Mumbai today?",   # not in the report: should say it doesn't know
]

for q in sample_questions:
    ask(q)

Q: What was the net profit for the year?
A: Based on the provided context, the profit for the year is 26,248 (for the current year) and 24,108 (for the previous year). Additionally, the text mentions that net profit increased to ₹26,233 crore on a consolidated basis and ₹27,234 crore on a standalone basis.
Sources: INFY_03062024153620_infosysar24.pdf (page 273); INFY_03062024153620_infosysar24.pdf (page 277); INFY_03062024153620_infosysar24.pdf (page 195); INFY_03062024153620_infosysar24.pdf (page 339); INFY_03062024153620_infosysar24.pdf (page 74)
------------------------------------------------------------
Q: Who is the Chief Executive Officer?
A: The Chief Executive Officer is Salil Parekh.
Sources: INFY_03062024153620_infosysar24.pdf (page 18); INFY_03062024153620_infosysar24.pdf (page 177); INFY_03062024153620_infosysar24.pdf (page 139); INFY_03062024153620_infosysar24.pdf (page 200); INFY_03062024153620_infosysar24.pdf (page 252)
--------------------------------------------------

## Step 11: Chat mode

An interactive loop for asking my own questions one after another. Type `quit` (or press Enter on an empty box) to stop. Each question uses one API request.

Questions work best when they are specific ("What was the net profit in FY24?" rather than "profit?"), use words that appear in the document, and ask one thing at a time.

In [ ]:
while True:
    q = input("Your question (or 'quit'): ").strip()
    if q.lower() in ("quit", "exit", ""):
        break
    ask(q)

Your question (or 'quit'): quit


## Step 12: Limitations, what I learned and next steps

### Limitations
- **Small test sets.** 12 development questions and 10 fresh ones, all drafted with AI assistance and checked against the report, on a single report. One question is worth 8 to 10 percentage points, so the scores are indicative, not proof.
- **Partly tuned.** I increased the context from 3 to 5 chunks after one development question failed, so the 12/12 is optimistic. The fresh-set result is the more honest number.
- **Rough auto-check.** It tests whether the expected value appears in the answer. An answer can contain it and still be partly wrong, so I read the answers too.
- **Tables and highlight boxes.** PDF tables and stat tiles become jumbled text, which is where retrieval and answers fail most often.
- **Scanned PDFs** are not supported (no OCR).
- **Free API limits.** The free Gemini tier has a small daily quota, and the API sometimes returns "high demand" errors.
- **Privacy.** Document text is sent to the API provider, so this should only be used with public or non-confidential documents. This report is public.

### What I learned
- Pure meaning search misses exact terms and numbers. Adding keyword search fixes much of that.
- A re-ranker is a cheap way to put the right chunk first.
- Once retrieval is good, the answer model becomes the bottleneck, especially on messy tables.
- Measuring with a test set showed things I would not have seen otherwise, and a second, untouched test set showed that my first scores were too optimistic.
- A system that refuses when it lacks evidence is more trustworthy than one that always answers.

### Next steps
1. **Improve retrieval for highlight boxes and tables** (extract tables separately, or index small stat-tile pages differently), then check the gain on a new, untouched question set.
2. **Add a second company's report** (for example TCS or HDFC Bank) with questions that need the right company.
3. **Wrap `ask()` in a small Gradio web app** so people can try it without Colab.

### Using other documents
Re-run Step 3 to upload more files, then re-run Steps 4 to 6 (chunking, embeddings, keyword index and re-ranker) so the new text is searchable. Steps 7 and 8 do not need re-running.